# HADL – full evaluation pipeline on a cloud GPU (Colab A100 / Kaggle 2×T4)

Runs steps 0–6 of `evaluation_pipeline/` end to end:
re-profile the time proxy on this GPU → costed candidate pool → subset selection → Qwen3-1.7B + LoRA training → evaluation → tables → figures.

**Colab:** Runtime → Change runtime type → A100 (or L4/T4). **Kaggle:** Accelerator → GPU T4 ×2, Internet on.

In [ ]:
# 1) Get the code (edit the URL/branch if you use a fork)
REPO_URL = "https://github.com/rahul-y-in/hadl_courseproject.git"
BRANCH = "main"
!git clone --depth 1 -b $BRANCH $REPO_URL hadl_courseproject
%cd hadl_courseproject/evaluation_pipeline
!nvidia-smi

In [ ]:
# 2) Dependencies (torch is pre-installed on Colab/Kaggle)
!pip -q install "transformers>=4.51" "peft>=0.12" "accelerate>=0.34" "datasets>=2.19" pyarrow scikit-learn scipy matplotlib pytest

In [ ]:
# 3) Quick sanity check: unit tests (CPU, ~10 s)
!python -m pytest -q tests

### 4) Run the whole pipeline
* `PROFILE=1` re-fits the training-time proxy on *this* GPU (the committed proxy was profiled on Apple MPS), ~5 min.
* Budget = `BUDGET_X_REAL` × proxy time of the real train split (1.0 ≈ “as much synthetic training time as real”).
* On **Kaggle 2×T4** add `NPROC=2 DTYPE=fp16 EVAL_BS=4`.
* Results are skipped if they already exist, so re-running after a disconnect resumes where it stopped.

In [ ]:
!PROFILE=1 SEEDS="42" BUDGET_X_REAL="1.0" METHODS="random quality_only proposed_knapsack" REFERENCE="real_only" bash run_pipeline.sh

Optional, more thorough (≈3× the GPU time): ablations + 3 seeds + 3 budgets

```
!SEEDS="42 43 44" BUDGET_X_REAL="0.5 1 2" METHODS=all REFERENCE=real_only STEPS="1 2 3 4 5 6" bash run_pipeline.sh
```

In [ ]:
# 5) Look at the results
import pandas as pd
from IPython.display import Markdown, Image, display
display(Markdown(open("outputs/results/results_table.md").read()))
display(Markdown(open("outputs/results/significance_vs_random.md").read()))
for name in ["qa_scores_test", "training_time_predicted_vs_actual", "proxy_step_parity",
             "selection_overview_b" + str(int(__import__('json').load(open('outputs/selections/budgets.json'))['budgets_sec'][0])),
             "pipeline_overhead", "training_loss_curves"]:
    try:
        display(Image(f"outputs/figures/{name}.png", width=900))
    except FileNotFoundError:
        print("missing", name)

In [ ]:
# 6) Download everything (tables, figures, predictions, adapters)
!zip -qr hadl_results.zip outputs -x "outputs/pool/*"
try:
    from google.colab import files; files.download("hadl_results.zip")
except ImportError:
    print("Kaggle: hadl_results.zip is in /kaggle/working/hadl_courseproject/evaluation_pipeline")